# Main results

Paired comparison of the 20 het/hom pairs from the cached per-network records. Writes the scoreboard and recruitment tables, the headline figures and the separability numbers.

In [ ]:
import sys
sys.path.append("../code")
import json
import numpy as np
import matplotlib.pyplot as plt
import common as C, stats as st, plots as P
P.apply()
het = C.load_records("het"); hom = C.load_records("hom")

## scoreboard

In [ ]:
V = lambda d, k: C.vals(d, k, C.SEEDS)
pcm = lambda d, m: np.array([d[s]["pc_curve"][d[s]["M_GRID"].index(m)] for s in C.SEEDS])

SCORE = [
    ("Task accuracy", V(het, "acc"), V(hom, "acc"), "{:.3f}"),
    ("Active neurons", V(het, "active"), V(hom, "active"), "{:.2f}"),
    ("Mean firing rate", V(het, "mean_rate"), V(hom, "mean_rate"), "{:.4f}"),
    ("Recurrent gain (spectral radius)", V(het, "gain"), V(hom, "gain"), "{:.3f}"),
    ("PCA n90 (1 ms)", V(het, "n90_1ms"), V(hom, "n90_1ms"), "{:.2f}"),
    ("PCA participation ratio (1 ms)", V(het, "pr_1ms"), V(hom, "pr_1ms"), "{:.2f}"),
    ("PCA top-5 variance", V(het, "top5_1ms"), V(hom, "top5_1ms"), "{:.3f}"),
    ("PCA top-10 variance", V(het, "top10_1ms"), V(hom, "top10_1ms"), "{:.3f}"),
    ("Linear decoder accuracy", V(het, "lin_decode"), V(hom, "lin_decode"), "{:.3f}"),
    ("PC-reconstruction @ m=4", pcm(het, 4), pcm(hom, 4), "{:.3f}"),
    ("PC-reconstruction @ m=16", pcm(het, 16), pcm(hom, 16), "{:.3f}"),
    ("PC-reconstruction @ m=32", pcm(het, 32), pcm(hom, 32), "{:.3f}"),
]


def md_table(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    return "\n".join(out)


rows, recs = [], []
for lbl, h, o, fmt in SCORE:
    r = st.paired_triple(h, o, lbl)
    recs.append(r)
    rows.append([lbl, fmt.format(r["het_mean"]), fmt.format(r["hom_mean"]), f"{r['delta']:+.3f}",
                 f"{r['dz']:+.2f}", P.fmt_p(r["t_p"]), P.fmt_p(r["w_p"], r["w_floor"]),
                 P.fmt_p(r["perm_p"], r["perm_floor"]), f"{r['wins']}/{r['n']}"])
for row, a in zip(rows, st.holm([r["perm_p"] for r in recs])):
    row.append(f"{a:.1e}" if a < 1e-3 else f"{a:.3f}")
scoreboard = {r["metric"]: r for r in recs}
print(md_table(["metric", "het", "hom", "delta", "dz", "paired-t p", "Wilcoxon p", "sign-flip p",
                "wins/n", "Holm p"], rows))

In [ ]:
import math


def sci(p):
    if p != p:
        return "n/a"
    if p >= 1e-3:
        return f"{p:.3f}"
    e = int(math.floor(math.log10(p)))
    return f"${p / 10 ** e:.1f}\\times10^{{{e}}}$"


SCORE_ORDER = [
    ("Task accuracy", "Task accuracy", "{:.3f}"),
    ("Linear decoder accuracy", "Linear decodability", "{:.3f}"),
    ("Recurrent gain (spectral radius)", "Recurrent gain", "{:.2f}"),
    ("PCA n90 (1 ms)", "PCA $n_{90}$", "{:.1f}"),
    ("PCA participation ratio (1 ms)", "Participation ratio", "{:.2f}"),
    ("PCA top-5 variance", "Top-5 variance", "{:.3f}"),
    ("PC-reconstruction @ m=16", "PC-reconstruction ($m{=}16$)", "{:.3f}"),
    ("PC-reconstruction @ m=32", "PC-reconstruction ($m{=}32$)", "{:.3f}"),
    ("Active neurons", "Active units", "{:.1f}"),
    ("Mean firing rate", "Mean firing rate", "{:.3f}"),
]
rows = []
for key, lbl, fmt in SCORE_ORDER:
    r = scoreboard[key]
    rows.append(f"{lbl} & {fmt.format(r['het_mean'])} & {fmt.format(r['hom_mean'])} & "
                f"{r['delta']:+.3f} & {r['dz']:+.2f} & {sci(r['t_p'])} & {sci(r['w_p'])} & "
                f"{sci(r['perm_p'])} & {r['wins']}/{r['n']} \\\\")

scoreboard_tex = r"""\begin{table}[H]
  \centering
  \small
  \setlength{\tabcolsep}{4pt}
  \caption{Within-seed paired comparison of the twenty heterogeneous/homogeneous pairs: per-arm means,
  paired difference, Cohen's $d_z$, paired $t$, Wilcoxon and exact sign-flip permutation $p$, and the
  number of seeds on which the heterogeneous network is higher.}
  \label{tab:scoreboard}
  \resizebox{\linewidth}{!}{%
  \begin{tabular}{lrrrrrrrr}
    \toprule
    Metric & het & hom & $\Delta$ & $d_z$ & paired $t$ $p$ & Wilcoxon $p$ & sign-flip $p$ & wins \\
    \midrule
""" + "\n    ".join(rows[:8]) + r"""
    \midrule
    """ + "\n    ".join(rows[8:]) + r"""
    \bottomrule
  \end{tabular}}
\end{table}
"""

_ra = scoreboard["Active neurons"]
recruit_tex = r"""\begin{table}[H]
  \centering
  \caption{Active hidden units (at least one spike on the test set), paired across the twenty seeds.}
  \label{tab:recruit}
  \resizebox{\linewidth}{!}{%
  \begin{tabular}{lrrrrrrrr}
    \toprule
    Quantity & het & hom & $\Delta$ & $d_z$ & paired $t$ $p$ & Wilcoxon $p$ & sign-flip $p$ & wins \\
    \midrule
    Active units & """ + (f"{_ra['het_mean']:.1f} & {_ra['hom_mean']:.1f} & {_ra['delta']:+.2f} & "
                          f"{_ra['dz']:+.2f} & {sci(_ra['t_p'])} & {sci(_ra['w_p'])} & "
                          f"{sci(_ra['perm_p'])} & {_ra['wins']}/{_ra['n']}") + r""" \\
    \bottomrule
  \end{tabular}}
\end{table}
"""

HDR = "% auto-generated by notebooks/02_main_results.ipynb; regenerate rather than edit.\n\n"
for name, tex in [("tab_scoreboard", scoreboard_tex), ("tab_recruitment", recruit_tex)]:
    p = C.TABDIR / f"{name}.tex"
    p.write_text(HDR + tex, encoding="utf-8")
    print(f"wrote {p.name} ({len(tex)} chars)")

## figures

In [ ]:
ANOM = {1004, 1007}  # long-tau hom seeds
acc_h, acc_o = V(het, "acc"), V(hom, "acc")
r_acc = scoreboard["Task accuracy"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
ax = axes[0]
for a, b in zip(acc_h, acc_o):
    ax.plot([0, 1], [a, b], color="#b0b0b0", lw=0.9, alpha=0.8, zorder=1)
ax.scatter(np.zeros(len(acc_h)), acc_h, color=P.HET, s=32, zorder=3, label=P.LABEL["het"])
ax.scatter(np.ones(len(acc_o)), acc_o, color=P.HOM, s=32, zorder=3, label=P.LABEL["hom"])
ax.set_xlim(-0.35, 1.35)
ax.set_xticks([0, 1])
ax.set_xticklabels([P.LABEL["het"], P.LABEL["hom"]])
P.chance_line(ax)
ax.set_ylabel("task accuracy")
ax.set_title(f"Paired accuracy, n = {len(acc_h)}")
ax.annotate(f"het wins {r_acc['wins']}/{r_acc['n']}\ndelta = {r_acc['delta']:+.3f}\n"
            f"sign-flip p = {P.fmt_p(r_acc['perm_p'], r_acc['perm_floor'])}",
            xy=(0.5, 0.06), xycoords="axes fraction", ha="center", fontsize=9, color=P.ACCENT)
ax.legend(loc="upper right")
d = acc_h - acc_o
o_ = np.argsort(d)
lbl = [str(C.SEEDS[i]) + ("*" if C.SEEDS[i] in ANOM else "") for i in o_]
axes[1].bar(range(len(d)), d[o_], color=[P.HET if v > 0 else P.HOM for v in d[o_]])
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xticks(range(len(d)))
axes[1].set_xticklabels(lbl, rotation=90, fontsize=7)
axes[1].set_ylabel("accuracy gain (het - hom)")
axes[1].set_xlabel("seed (sorted by gain; * = long-tau hom)")
axes[1].set_title("Per-seed accuracy gain")
P.save(fig, C.FIGDIR / "F4_1_paired_accuracy.png")

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
allt = np.concatenate([np.array(het[s]["tau_all_ms"]) for s in C.SEEDS])
bins = np.logspace(np.log10(3), np.log10(210), 40)
ax.hist(allt, bins=bins, color=P.HET, alpha=0.75, label=f"het, pooled per-neuron tau (n = {allt.size})")
for s in C.SEEDS:
    ax.axvline(hom[s]["tau_median_ms"], color=P.HOM, alpha=0.5, lw=1.2)
ax.axvline(np.nan, color=P.HOM, lw=1.2, label="hom, single learned tau (one line per seed)")  # legend proxy
ax.axvline(199.5, color=P.CHANCE, ls="--", lw=1.4, label="clip ceiling 199.5 ms")
ax.axvline(3.26, color=P.CHANCE, ls=":", lw=1.4, label="clip floor 3.26 ms")
ax.set_xscale("log")
ax.set_xlabel("membrane time constant (ms, log scale)")
ax.set_ylabel("number of hidden units")
ax.set_title("Learned time-constant solutions")
ax.legend(fontsize=8)
P.save(fig, C.FIGDIR / "F4_2_tau_solutions.png")

In [ ]:
act_h, act_o = V(het, "active"), V(hom, "active")
r_act = scoreboard["Active neurons"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
d_act = act_h - act_o
d_sorted = d_act[np.argsort(d_act)]
axes[0].bar(np.arange(len(d_sorted)), d_sorted, color=[P.HET if v > 0 else P.HOM for v in d_sorted], zorder=3)
axes[0].axhline(0, color="0.25", lw=1.0, zorder=4)
axes[0].axhline(d_act.mean(), color=P.ACCENT, ls="--", lw=1.3, zorder=5,
                label=f"mean {d_act.mean():+.2f} ($p$ = {r_act['perm_p']:.3f})")
axes[0].set_xlabel("seed, sorted by difference")
axes[0].set_xticks(np.arange(0, len(d_sorted), 4))
axes[0].set_ylabel("active units, het $-$ hom")
axes[0].set_title("Recruitment difference per seed")
axes[0].legend(loc="upper left", fontsize=9)
xs = np.concatenate([act_h, act_o])
ys = np.concatenate([acc_h, acc_o])
axes[1].scatter(xs, ys, c=[P.HET] * len(act_h) + [P.HOM] * len(act_o), s=30, zorder=3)
rr = np.corrcoef(xs, ys)[0, 1]
z = np.polyfit(xs, ys, 1)
xg = np.linspace(xs.min(), xs.max(), 50)
axes[1].plot(xg, np.polyval(z, xg), color=P.ACCENT, ls="--", lw=1.3, label=f"r = {rr:.2f}")
P.chance_line(axes[1])
axes[1].set_xlabel("active hidden units")
axes[1].set_ylabel("task accuracy")
axes[1].set_title("Active neurons vs accuracy")
axes[1].legend()
P.save(fig, C.FIGDIR / "F4_3_recruitment.png")

In [ ]:
E = {a: np.stack([np.array(d[s]["evr_1ms"]) for s in C.SEEDS]) for a, d in [("het", het), ("hom", hom)]}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
x = np.arange(1, 65)
for a in ["het", "hom"]:
    m, sd = E[a].mean(0), E[a].std(0)
    P.band(axes[0], x, np.clip(m, 1e-7, None), sd, P.COL[a], label=P.LABEL[a], marker="o", ms=2.5)
    cm = np.cumsum(E[a], 1)
    P.band(axes[1], x, cm.mean(0), cm.std(0), P.COL[a], label=P.LABEL[a], marker="o", ms=2.5)
axes[0].set_yscale("log")
axes[0].set_ylim(1e-6, 1.5)
axes[0].set_xlabel("principal component")
axes[0].set_ylabel("explained variance ratio")
axes[0].set_title("Variance decay, time-resolved [(trials x time) x 64] at 1 ms")
axes[0].legend()
axes[1].axhline(0.9, color=P.CHANCE, ls=":", lw=1.2)
axes[1].set_xlabel("principal component")
axes[1].set_ylabel("cumulative explained variance")
n9h, n9o = V(het, "n90_1ms"), V(hom, "n90_1ms")
axes[1].set_title(f"Cumulative variance (n90: het {n9h.mean():.1f}, hom {n9o.mean():.1f})")
axes[1].legend()
P.save(fig, C.FIGDIR / "F4_7_pca_scree.png")

In [ ]:
MG = het[C.SEEDS[0]]["M_GRID"]
PC = {a: np.stack([np.array(d[s]["pc_curve"]) for s in C.SEEDS]) for a, d in [("het", het), ("hom", hom)]}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for a in ["het", "hom"]:
    P.band(axes[0], MG, PC[a].mean(0), PC[a].std(0), P.COL[a], label=P.LABEL[a], marker="o", ms=4)
P.chance_line(axes[0])
axes[0].set_xscale("log", base=2)
axes[0].set_xticks(MG)
axes[0].set_xticklabels(MG)
axes[0].set_xlabel("number of principal components in the reconstruction")
axes[0].set_ylabel("task accuracy through the real readout")
axes[0].set_title("PC-reconstruction accuracy, 1 ms, all 2,264 test trials")
axes[0].legend(loc="lower right")
gapc = PC["het"].mean(0) - PC["hom"].mean(0)
axes[1].plot(MG, gapc, color=P.ACCENT, marker="o", ms=4)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xscale("log", base=2)
axes[1].set_xticks(MG)
axes[1].set_xticklabels(MG)
axes[1].set_xlabel("number of principal components")
axes[1].set_ylabel("het advantage (het - hom)")
# durable crossover: last m at which hom still leads, one grid step up
neg_idx = [i for i in range(len(MG)) if gapc[i] < 0]
cross = MG[max(neg_idx) + 1] if neg_idx and max(neg_idx) + 1 < len(MG) else MG[0]
if cross:
    axes[1].axvline(cross, color=P.CHANCE, ls="--", lw=1.2)
axes[1].set_title("Het minus hom accuracy vs number of PCs")
P.save(fig, C.FIGDIR / "F4_8_pc_reconstruction.png")

In [ ]:
# capacity summary is written by 03_geometry.ipynb; rerun this cell after it
cap = json.load(open(C.RESULTS / "capacity_corr_n20_summary.json"))["paired"]
order = [("alpha_M", "capacity $\\alpha_M$"), ("R_M", "radius $R_M$"),
         ("D_M", "dimension $D_M$"), ("center_corr", "centroid corr.")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, (setk, nice) in zip(axes, [("A_8cond", "8 conditions"), ("B_20cond", "20 conditions")]):
    dz = [cap[setk][k]["dz"] for k, _ in order]
    wins = [f"{cap[setk][k]['wins']}/{cap[setk][k]['n']}" for k, _ in order]
    y = np.arange(len(order))[::-1]
    ax.barh(y, dz, color=[P.HET if v > 0 else P.HOM for v in dz], height=0.6)
    ax.axvline(0, color="#444", lw=1)
    for yi, v, w in zip(y, dz, wins):
        ax.text(v + (0.06 if v > 0 else -0.06), yi, w, va="center",
                ha="left" if v > 0 else "right", fontsize=8)
    ax.set_yticks(y)
    ax.set_yticklabels([n for _, n in order])
    ax.set_xlabel("paired effect size $d_z$ (het $-$ hom)")
    ax.set_title(nice)
    ax.set_xlim(-2.2, 2.2)
fig.suptitle("Manifold radius, dimension and center correlation, het vs hom",
             fontweight="bold", y=1.02)
P.save(fig, C.FIGDIR / "F4_18_capacity_components.png")

## separability vs accuracy

In [ ]:
from scipy.stats import pearsonr

scaled = C.load_records("scaled", C.SEEDS_ORIG)
sep_h, sep_o = V(het, "lin_decode"), V(hom, "lin_decode")
sep_s, acc_s = C.vals(scaled, "lin_decode"), C.vals(scaled, "acc")
x40 = np.concatenate([sep_h, sep_o])
y40 = np.concatenate([acc_h, acc_o])
r40, p40 = pearsonr(x40, y40)
r50, p50 = pearsonr(np.concatenate([x40, sep_s]), np.concatenate([y40, acc_s]))
arm = {"het": pearsonr(sep_h, acc_h), "hom": pearsonr(sep_o, acc_o), "scaled": pearsonr(sep_s, acc_s)}
print(f"r, 40 learned nets:            {r40:+.3f}  p = {p40:.2e}")
print(f"r, 50 nets incl. scaled-init:  {r50:+.3f}  p = {p50:.2e}")
for a, (r_, p_) in arm.items():
    print(f"r within {a}: {r_:+.3f}  p = {p_:.2e}")
out = {
    "metric": "lin_decode, 5-fold linear decode of class from the trial rate code",
    "learned_40": {"r": float(r40), "p": float(p40), "n": int(len(x40))},
    "all_50_with_scaled": {"r": float(r50), "p": float(p50), "n": int(len(x40) + len(sep_s))},
    "within_arm": {a: {"r": float(r_), "p": float(p_), "n": int(len(sep_s) if a == "scaled" else len(sep_h))}
                   for a, (r_, p_) in arm.items()},
}
json.dump(out, open(C.RESULTS / "separability.json", "w"), indent=1)

fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.scatter(sep_h, acc_h, color=P.HET, s=30, zorder=3, label=P.LABEL["het"])
ax.scatter(sep_o, acc_o, color=P.HOM, s=30, zorder=3, label=P.LABEL["hom"])
z = np.polyfit(x40, y40, 1)
xg = np.linspace(x40.min(), x40.max(), 50)
ax.plot(xg, np.polyval(z, xg), color=P.ACCENT, ls="--", lw=1.3, label=f"r = {r40:.2f}")
P.chance_line(ax)
ax.set_xlabel("linear decodability of class from the rate code (5-fold)")
ax.set_ylabel("task accuracy")
ax.set_title("Linear decodability vs accuracy")
ax.legend(loc="lower right")
P.save(fig, C.FIGDIR / "F4_9_separability_vs_accuracy.png")